<a href="https://colab.research.google.com/github/roman-malter/cloud-automation-portfolio/blob/main/azure_nsg_validator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project 4: Cloud Network Security Group (NSG) Firewall Validator

### Project Overview
This automated cloud security tool scans network firewall configurations (Network Security Groups) within an enterprise infrastructure. It programmatically identifies high-risk vulnerabilities, specifically targeting inbound rules that expose management ports—such as SSH (Port 22) and RDP (Port 3389)—to the public internet. The tool ensures compliance with the principle of least privilege.

### Skills Demonstrated
* Security policy enforcement via programmatic loops
* Firewall rule analysis and public network exposure mapping
* Automated vulnerability auditing and network risk mitigation


In [1]:
# Simulated Azure Network Security Group (NSG) Inbound Firewall Rules
azure_nsg_rules = [
    {"rule_name": "Allow-HTTP-Public", "port": 80, "source_address": "0.0.0.0/0", "action": "Allow", "description": "Public Web Traffic"},
    {"rule_name": "Allow-SSH-AdminCorp", "port": 22, "source_address": "192.168.1.50/32", "action": "Allow", "description": "Secure Corporate Office Access"},
    {"rule_name": "Allow-RDP-Anywhere", "port": 3389, "source_address": "0.0.0.0/0", "action": "Allow", "description": "Temporary Developer Remote Access"},
    {"rule_name": "Allow-SSH-Anywhere", "port": 22, "source_address": "*", "action": "Allow", "description": "Default Management Port Access"},
    {"rule_name": "Deny-All-Inbound", "port": "*", "source_address": "0.0.0.0/0", "action": "Deny", "description": "Baseline Security Cleanup Rule"}
]

print("🛡️ --- STARTING AUTOMATED CLOUD FIREWALL AUDIT --- 🛡️\n")

critical_vulnerabilities = 0
safe_rules = 0

# Loop through each firewall rule to analyze the network exposure
for rule in azure_nsg_rules:
    print(f"Analyzing Rule: {rule['rule_name']} | Port: {rule['port']} | Action: {rule['action']}")

    # Check if the rule allows traffic from the open internet
    is_public = rule['source_address'] == "0.0.0.0/0" or rule['source_address'] == "*"
    is_allowed = rule['action'] == "Allow"

    # AUDIT RULE: Exposing administrative ports (22 or 3389) to the public internet is a CRITICAL risk
    if is_allowed and is_public:
        if rule['port'] == 22 or rule['port'] == 3389:
            print(f"  🚨 CRITICAL RISK: Management port {rule['port']} is WIDE OPEN to the public internet!")
            print(f"  👉 Mitigation: Restrict source address to an explicit corporate IP or deploy Azure Bastion.")
            critical_vulnerabilities += 1
            print("-" * 75)
            continue

    # If a web port (80/443) is open to the public, it's normal behavior for a website
    if is_allowed and is_public and rule['port'] == 80:
        print(f"  ✅ APPROVED: Public web port verified. Standard application traffic allowed.")
        safe_rules += 1
    # If management ports are restricted to a single corporate IP, it's acceptable
    elif is_allowed and not is_public and (rule['port'] == 22 or rule['port'] == 3389):
        print(f"  ✅ SECURE: Management port restricted to safe corporate source IP address ({rule['source_address']}).")
        safe_rules += 1
    else:
        print(f"  ℹ️ COMPLIANT: Rule meets basic security perimeter standards.")
        safe_rules += 1

    print("-" * 75)

# Generate Final Network Security Report
print("\n📋 --- NETWORK AUDIT SUMMARY REPORT ---")
print(f"Compliant/Secure Rules: {safe_rules}")
print(f"Critical Perimeter Risks: {critical_vulnerabilities}")

if critical_vulnerabilities > 0:
    print("STATUS: FAILED | Unauthorized public exposure detected. Network perimeter is vulnerable to attack.")
else:
    print("STATUS: PASSED | Network perimeter security meets compliance thresholds.")


🛡️ --- STARTING AUTOMATED CLOUD FIREWALL AUDIT --- 🛡️

Analyzing Rule: Allow-HTTP-Public | Port: 80 | Action: Allow
  ✅ APPROVED: Public web port verified. Standard application traffic allowed.
---------------------------------------------------------------------------
Analyzing Rule: Allow-SSH-AdminCorp | Port: 22 | Action: Allow
  ✅ SECURE: Management port restricted to safe corporate source IP address (192.168.1.50/32).
---------------------------------------------------------------------------
Analyzing Rule: Allow-RDP-Anywhere | Port: 3389 | Action: Allow
  🚨 CRITICAL RISK: Management port 3389 is WIDE OPEN to the public internet!
  👉 Mitigation: Restrict source address to an explicit corporate IP or deploy Azure Bastion.
---------------------------------------------------------------------------
Analyzing Rule: Allow-SSH-Anywhere | Port: 22 | Action: Allow
  🚨 CRITICAL RISK: Management port 22 is WIDE OPEN to the public internet!
  👉 Mitigation: Restrict source address to an expl